# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 7: Fine-Tuning LLMs</font>

# <font color="#003660">Notebook 3: Designing Training Data</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know how to design training data in accordance with HuggingFace templates.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:


* [HF LLM Course](https://huggingface.co/learn/llm-course/chapter11/2)
* [HF ChatTemplates](https://huggingface.co/docs/transformers/chat_templating)

## Installing and Setup

In [ ]:
!pip install -U datasets transformers accelerate bitsandbytes peft trl wandb

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# helper function
def generate(messages, model):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)[0]
    return response

In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_type=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

In [ ]:
messages = [{"role": "user", "content": "What is the meaning of life?"}, {"role": "assistant", "content": "The meaning of life is to serve as an AI assistant."}]
print(
    tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
)

This is all automatically applied by what is called a [ChatTemplate]() in HuggingFace. Let's look into the ChatTemplate below.

In [ ]:
print(tokenizer.chat_template)

Wow, looks complex. I will tell some basics in the following. A complete guide can be found here.

What happens if we add a system prompt.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the meaning of life?"},
    {"role": "assistant", "content": "The meaning of life is to serve as an AI assistant."}
]
print(
    tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )
)

## Tool Calling - The Basic of agents

What about Tools?

In [ ]:
def get_current_temperature(location: str):
    """
    Gets the temperature at a given location.

    Args:
        location: The location to get the temperature for
    """
    return 22.0  # bug: Sometimes the temperature is not 22. low priority

tools = [get_current_temperature]

chat = [
    {"role": "user", "content": "Hey, what's the weather like in Paris right now?"}
]

tool_prompt = tokenizer.apply_chat_template(
    chat,
    tools=tools,
    tokenize=False
)
print(tool_prompt)


Usually, we hope the LLm to be call a tool like below.

In [ ]:
message = {
    "role": "assistant",
    "tool_calls": [
        {
            "type": "function",
             "function": {
                 "name": "get_current_temperature",
                 "arguments": {
                     "location": "Paris, France"
                }
            }
        }
    ]
}
chat.append(message)

In [ ]:
tool_prompt = tokenizer.apply_chat_template(
    chat,
    tools=tools,
    tokenize=False
)
print(tool_prompt)

This tool call will then be identified during execution with a smart regex.

Then the tool will be executed and return the answer like below.

In [ ]:
message = {
    "role": "tool",
    "name": "get_current_temperature",
    "content": "22.0"
}
chat.append(message)

In [ ]:
tool_prompt = tokenizer.apply_chat_template(
    chat,
    tools=tools,
    tokenize=False
)
print(tool_prompt)

Based on this, the LLM will answer.

In [ ]:
message = {
    "role": "assistant",
    "content": "The temperature in Paris is currently 22.0 °C."
}
chat.append(message)

In [ ]:
tool_prompt = tokenizer.apply_chat_template(
    chat,
    tools=tools,
    tokenize=False
)
print(tool_prompt)

## Your Turn - How to do this for RAG

In [ ]:
# TODO: Implement RAG Template